# NHD Feature Matcher
Match fracking water sources to NHD features using fuzzy name matching
constrained by proximity to well coordinates.

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')
from rapidfuzz import fuzz
from math import radians, cos, sin, asin, sqrt

print('Loading NHD...')
nhd_fl = gpd.read_file('data/NHD_PA_named.gpkg', layer='NHDFlowline')
nhd_wb = gpd.read_file('data/NHD_PA_named.gpkg', layer='NHDWaterbody')
nhd = pd.concat([nhd_fl, nhd_wb], ignore_index=True)
print(f'  Flowlines:   {len(nhd_fl):,}')
print(f'  Waterbodies: {len(nhd_wb):,}')
print(f'  Total NHD:   {len(nhd):,}')

In [ ]:
SKINNY_PATH = r'G:\My Drive\production\repos\openFF_data_2026_04_03\skinny_df.parquet'

jct    = pd.read_parquet('data/well_junction_table.parquet')
master = pd.read_parquet('data/FPW_master_water_source.parquet')

well_coords = (
    pd.read_parquet(SKINNY_PATH, columns=['api10', 'bgLatitude', 'bgLongitude'])
    .groupby('api10', as_index=False).first()
    .rename(columns={'bgLatitude': 'well_lat', 'bgLongitude': 'well_lon'})
)
jct = jct.merge(well_coords, on='api10', how='left')

RULES = [
    ('reuse',          r'recycl|reuse|re-use|flowback|produced water|rain\s*water'),
    ('interconnection',r'\bintc\b|\btap\b|\bvending\b|\bvault\b|\bhydrant\b'
                       r'|\bauthority\b|\bmunicipal\b|water\s+co[^u]|\bmeter\b'
                       r'|public\s+water|\bpwsid\b|water\s+system|water\s+service'
                       r'|water\s+company|interconnect'),
    ('groundwater',    r'\bwell\b|\bspring\b|\baquifer\b|ground\s*water'),
    ('impoundment',    r'impoundment|\bpit\b|frac\s+pond'),
    ('surface_direct', r'creek|river|\brun\b|stream|\blake\b|\bpond\b|reservoir'
                       r'|brook|\bbranch\b|\bfork\b|hollow|\bdam\b|hatchery'),
    ('srbc_only',      r'srbc\s+docket'),
    ('dont_know',     r'\bSWW\b|\bSPWA\b|\bSWPA\b|\bNKWA\b|\bMAWC\b|\bMANK\b'
                      r'|\bAqua\b|\bWI\b|\bbrine\b|quarry|limestone\s+mine'
                      r'|\bClermont\b|\bAWS\b'),
]
def classify(name):
    if pd.isna(name): return 'no_source'
    for label, pat in RULES:
        if re.search(pat, str(name), re.IGNORECASE): return label
    return 'ambiguous'

jct['source_type'] = jct['planSource'].apply(classify)
jct['has_srbc']    = jct['planSource'].str.contains(r'srbc\s+docket', case=False, na=False)

cands = jct[jct['source_type'].isin(['surface_direct','impoundment','srbc_only'])].copy()
valid_id = cands['site_ID'].notna() & ~cands['site_ID'].isin(['','ambiguous'])
candidates = pd.concat([
    cands[valid_id].drop_duplicates('site_ID')[['site_ID','planSource','site_clean','source_type','has_srbc']],
    cands[~valid_id].drop_duplicates('planSource')[['site_ID','planSource','site_clean','source_type','has_srbc']],
], ignore_index=True)

candidates = candidates.merge(master[['site_ID','site_name','Latitude','Longitude']], on='site_ID', how='left')
well_proxy = (cands.groupby('planSource')[['well_lat','well_lon']].median().reset_index()
              .rename(columns={'well_lat':'proxy_lat','well_lon':'proxy_lon'}))
candidates = candidates.merge(well_proxy, on='planSource', how='left')
candidates['coord_lat'] = candidates['Latitude'].fillna(candidates['proxy_lat'])
candidates['coord_lon'] = candidates['Longitude'].fillna(candidates['proxy_lon'])
candidates = candidates[candidates['coord_lat'].notna()].reset_index(drop=True)
print(f'Candidate sources with coordinates: {len(candidates):,}')

## Name extraction and normalization

In [ ]:
WATER_TERMS = (
    r'\b(?:creek|river|run|stream|lake|pond|reservoir|brook|branch|'
    r'fork|hollow|dam|hatchery|spring|susquehanna|wyalusing)\b'
)

_ABBREVS = [
    (r'\bN\.?\s+',  'North '),  (r'\bS\.?\s+',  'South '),
    (r'\bE\.?\s+',  'East '),   (r'\bW\.?\s+',  'West '),
    (r'\bNE\.?\s+', 'Northeast '), (r'\bNW\.?\s+', 'Northwest '),
    (r'\bSE\.?\s+', 'Southeast '), (r'\bSW\.?\s+', 'Southwest '),
    (r'\bBr\.?\s+', 'Branch '), (r'\bFk\.?\s+', 'Fork '),
    (r'\bCk\.?\b',  'Creek'),   (r'\bCr\.?\b',  'Creek'),
    (r'\bUnt\.?\b', 'Unnamed Tributary'), (r'\bUNT\.?\b', 'Unnamed Tributary'),
]

def normalize_name(s):
    if not s or pd.isna(s): return ''
    s = str(s)
    for pat, repl in _ABBREVS:
        s = re.sub(pat, repl, s, flags=re.IGNORECASE)
    s = re.sub(r'[^a-z\s]', ' ', s.lower())
    return re.sub(r'\s+', ' ', s).strip()

_OP_PREFIXES = {
    'cabot','coterra','bkv','sfgs','sgfs','bnr','carrizo','repsol',
    'chesapeake','eqt','range','swn','talisman','cnx','well','field',
    'diaz','garrison',
}

_DELIM = re.compile(r',|\s+-\s+|\s+@\s+')

def extract_search_name(planSource):
    if pd.isna(planSource): return ''
    s = str(planSource)
    s = re.sub(r'[\[\(]SRBC[^\]\)]*[\]\)]', '', s, flags=re.IGNORECASE)
    s = re.sub(r'\bSRBC\s+Docket\s+(?:No\.|Number)?\s*\d[\d\-]*', '', s, flags=re.IGNORECASE)
    s_np = re.sub(r'\([^)]*\)', '', s)
    s_np = re.sub(r'\b(WV|PA|OH|NY|MD|NJ)\b', '', s_np)
    s_np = re.sub(r'\b(source|station|rhl|wwp|wwd|dcnr|swpa)\b.*$', '', s_np, flags=re.IGNORECASE)
    s_np = s_np.strip(' ,.-')

    feature = None
    for seg in reversed([seg.strip() for seg in _DELIM.split(s_np)]):
        if re.search(WATER_TERMS, seg, re.IGNORECASE):
            feature = seg.strip(' -.')
            break
    if feature is None:
        for paren in re.findall(r'\(([^)]+)\)', s):
            if re.search(WATER_TERMS, paren, re.IGNORECASE):
                feature = paren.strip()
                break
    if feature is None and re.search(WATER_TERMS, s_np, re.IGNORECASE):
        feature = s_np
    if feature is None:
        return ''

    m_last = None
    for m in re.finditer(WATER_TERMS, feature, re.IGNORECASE):
        m_last = m
    if m_last:
        feature = feature[:m_last.end()].strip(' -.')

    words = feature.split()
    while words:
        tok = re.sub(r'[\-\.\#\d]', '', words[0]).lower()
        if tok in _OP_PREFIXES or tok == '':
            words = words[1:]
        else:
            break
    return ' '.join(words).strip()


candidates['search_name']      = candidates['planSource'].apply(extract_search_name)
candidates['search_name_norm'] = candidates['search_name'].apply(normalize_name)
nhd['gnis_norm'] = nhd['gnis_name'].apply(normalize_name)
nhd['cx'] = nhd.geometry.centroid.x
nhd['cy'] = nhd.geometry.centroid.y

matched_cands = candidates[candidates['search_name_norm'] != ''].copy()
print(f'Candidates with extractable name: {len(matched_cands):,} / {len(candidates):,}')
print()
checks = matched_cands[matched_cands['planSource'].str.contains(
    'Susquehanna Gas|Rock Run|Columbia Unt', case=False, na=False)]
print(checks[['planSource','search_name']].to_string())

## Spatial + fuzzy matcher

In [ ]:
_DEG_LAT = 1 / 111.0
_DEG_LON = 1 / 85.0

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371
    dlat, dlon = radians(lat2-lat1), radians(lon2-lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    return 2*R*asin(sqrt(a))


def find_nhd_match(search_norm, lat, lon, radius_km=50, top_n=3):
    if not search_norm:
        return []
    dlat, dlon = radius_km*_DEG_LAT, radius_km*_DEG_LON
    nearby = nhd[
        (nhd['gnis_norm'] != '') &
        nhd['cy'].between(lat-dlat, lat+dlat) &
        nhd['cx'].between(lon-dlon, lon+dlon)
    ]
    if nearby.empty:
        return []
    scores = nearby['gnis_norm'].apply(lambda n: fuzz.token_sort_ratio(search_norm, n))
    top_idx = scores.nlargest(top_n).index
    results = []
    for idx in top_idx:
        row = nhd.loc[idx]
        dist = haversine_km(lat, lon, row['cy'], row['cx'])
        results.append({
            'nhd_id':    row['permanent_identifier'],
            'nhd_name':  row['gnis_name'],
            'nhd_ftype': row['ftype'],
            'nhd_layer': row['layer'],
            'score':     scores[idx],
            'dist_km':   round(dist, 1),
        })
    return sorted(results, key=lambda r: (-r['score'], r['dist_km']))


print(f'Running matcher on {len(matched_cands):,} candidates...')
records = []
for _, row in matched_cands.iterrows():
    hits = find_nhd_match(row['search_name_norm'], row['coord_lat'], row['coord_lon'])
    best = hits[0] if hits else {}
    records.append({
        'planSource':  row['planSource'],
        'source_type': row['source_type'],
        'has_srbc':    row['has_srbc'],
        'search_name': row['search_name'],
        'coord_lat':   row['coord_lat'],
        'coord_lon':   row['coord_lon'],
        'nhd_id':      best.get('nhd_id'),
        'nhd_name':    best.get('nhd_name'),
        'nhd_ftype':   best.get('nhd_ftype'),
        'nhd_layer':   best.get('nhd_layer'),
        'score':       best.get('score'),
        'dist_km':     best.get('dist_km'),
    })

results = pd.DataFrame(records)
print('Done.')
print(f'Matched (score >= 80): {(results["score"] >= 80).sum():,}')
print(f'Matched (score >= 60): {(results["score"] >= 60).sum():,}')
print(f'No NHD feature nearby: {results["nhd_name"].isna().sum():,}')

## Results review

In [ ]:
bins   = [0, 50, 60, 70, 80, 90, 100]
labels = ['<50','50-59','60-69','70-79','80-89','90-100']
results['score_bin'] = pd.cut(results['score'], bins=bins, labels=labels, right=True)
print('Score distribution:')
print(results['score_bin'].value_counts().sort_index().to_string())
print()
print('Median distance by score bin (km):')
print(results.groupby('score_bin', observed=True)['dist_km'].median().to_string())

In [ ]:
high = results[results['score'] >= 80].sort_values('score', ascending=False)
print(f'High-confidence matches (score >= 80): {len(high)}')
print(high[['search_name','nhd_name','score','dist_km','nhd_ftype','nhd_layer']]
      .head(30).to_string())

In [ ]:
low = results[results['score'].isna() | (results['score'] < 60)]
print(f'Low/no match (score < 60 or no nearby feature): {len(low)}')
print(low[['search_name','nhd_name','score','dist_km']].head(30).to_string())

In [ ]:
results.to_parquet('data/nhd_match_results.parquet', index=False)
print('Saved: data/nhd_match_results.parquet')

## Join match results back to junction table
Attach NHD match to every junction-table row to report volume by match confidence.

In [ ]:
jct_matched = jct.merge(
    results[['planSource','search_name','nhd_id','nhd_name',
             'nhd_ftype','nhd_layer','score','dist_km']],
    on='planSource', how='left'
)

def confidence_tier(score):
    if pd.isna(score):  return 'unmatched'
    if score >= 90:     return 'high (>=90)'
    if score >= 80:     return 'good (80-89)'
    if score >= 60:     return 'fair (60-79)'
    return              'low (<60)'

jct_matched['match_tier'] = jct_matched['score'].apply(confidence_tier)
tier_order = ['high (>=90)', 'good (80-89)', 'fair (60-79)', 'low (<60)', 'unmatched']

def vol_table(df):
    return (
        df.groupby('match_tier')['volume']
        .agg(records='count', volume_gal='sum')
        .assign(
            volume_Mgal = lambda d: (d['volume_gal'] / 1e6).round(1),
            pct_volume  = lambda d: (d['volume_gal'] / d['volume_gal'].sum() * 100).round(1),
        )
        .reindex(tier_order)
        [['records', 'volume_Mgal', 'pct_volume']]
    )

print('=== All junction rows ===')
print(vol_table(jct_matched).to_string())
print()
cand_rows = jct_matched[jct_matched['source_type'].isin(['surface_direct','impoundment','srbc_only'])]
print('=== Surface / impoundment / SRBC candidates only ===')
print(vol_table(cand_rows).to_string())

In [ ]:
jct_matched.to_parquet('data/junction_nhd_matched.parquet', index=False)
print(f'Saved: data/junction_nhd_matched.parquet  ({len(jct_matched):,} rows, {len(jct_matched.columns)} columns)')